# Multiverse basics

To conduct a multiverse analysis, the forking paths must be specified in a dictionary. Options can contain:

* strings
* numerical values
* boolean values
* comet dFC methods
* comet and bct graph measures
* any kind of function

In [1]:
from comet.multiverse import Multiverse

forking_paths = {
    "strings": ["Hello", "world"],
    "numbers": [3.14, 4, 5.2],
    "booleans": [True, False],
    "dfc_measures": [
        {"name": "LeiDA", "func": "comet.connectivity.LeiDA(time_series=ts).estimate()"},
        {"name": "JC11", "func": "comet.connectivity.Jackknife(time_series=ts, windowsize=11).estimate()"}],
    "graph_measures": [
        {"name": "efficiency", "func": "comet.graph.efficiency(G, local= False)"},
        {"name": "clustering", "func": "comet.graph.avg_clustering_onella(G)"}],
}

With the decisions and options defined, an analysis template has to be specified. This is similar to a standard analysis pipeline with three additional requirements:

* The template is required to be encapsulated in a dedicated function
* Required imports need to be within the template function
* Decision points need to be specified in double brackets: `{{decision}}`

In this brief example, the corresponding string, number, and boolean decision will be printed in each universe. Then, connectivity will be estimated with the corresponding dFC method, and a graph measure (local efficiency or clustering) is calculated:

In [2]:
def analysis_template():
    import comet

    print("Decision 1:", {{strings}})
    print("Decision 2:", {{numbers}})
    print("Decision 3:", {{booleans}})

    # Load example data and calculate dFC
    ts = comet.utils.load_example()
    dfc = {{dfc_measures}}

    # Calculate graph measure
    graph_measure = []
    for i in range(dfc.shape[2]):
        G = dfc[:, :, i]
        G = comet.graph.handle_negative_weights(G, type="absolute")
        G = comet.graph.threshold(G, type="density", density=0.5)
        G = comet.graph.binarise(G)
        gm = {{graph_measures}}
        graph_measure.append(gm)

    # Save the results
    result = {"graph_measure": graph_measure, "number": {{numbers}}}
    comet.utils.save_universe_results(result)

The forking paths dictionary defines 5 decision points consisting of 2 options each. Thus, the resulting multiverse will contain 2⁵=32 universes. A `Multiverse` object has to be created and can then be used to create, run, summarize, and visualize the multiverse.

* `mverse.create()` will generate Python scripts for all 32 universes. These scripts will be saved in a newly created `example_multiverse/` folder
* `mverse.summary()` will print the decisions for every universe. This information is also available as a .csv in the `example_multiverse/results/` folder
* `mverse.run()` will either run all or individual universes. If the computational resources allow for it, this can be parallelized by using e.g. `mverse.run(parallel=4)`

In [3]:
mverse = Multiverse(name="example_mv_basics")
mverse.create(analysis_template, forking_paths)
mverse.summary()

,Universe,Decision 1,Value 1,Decision 2,Value 2,Decision 3,Value 3,Decision 4,Value 4,Decision 5,Value 5
0,Universe_1,strings,Hello,numbers,3.14,booleans,True,dfc_measures,LeiDA,graph_measures,efficiency
1,Universe_2,strings,Hello,numbers,3.14,booleans,True,dfc_measures,LeiDA,graph_measures,clustering
2,Universe_3,strings,Hello,numbers,3.14,booleans,True,dfc_measures,JC11,graph_measures,efficiency
3,Universe_4,strings,Hello,numbers,3.14,booleans,True,dfc_measures,JC11,graph_measures,clustering
4,Universe_5,strings,Hello,numbers,3.14,booleans,False,dfc_measures,LeiDA,graph_measures,efficiency
5,Universe_6,strings,Hello,numbers,3.14,booleans,False,dfc_measures,LeiDA,graph_measures,clustering
6,Universe_7,strings,Hello,numbers,3.14,booleans,False,dfc_measures,JC11,graph_measures,efficiency
7,Universe_8,strings,Hello,numbers,3.14,booleans,False,dfc_measures,JC11,graph_measures,clustering
8,Universe_9,strings,Hello,numbers,4.00,booleans,True,dfc_measures,LeiDA,graph_measures,efficiency
9,Universe_10,strings,Hello,numbers,4.00,booleans,True,dfc_measures,LeiDA,graph_measures,clustering


You can now run individual universes by specifying a number, or run all of them (parallelization is also supported). This example will only evaluate the first two universes (`universe=[1,2]`).

In [4]:
mverse.run(universe=[1,2], parallel=2)

Starting analysis for universe(s): [1, 2]...


Performing multiverse analysis::   0%|          | 0/2 [00:00<?, ?it/s]

Decision 1: Hello
Decision 2: 3.14
Decision 3: True
Decision 1: Hello
Decision 2: 3.14
Decision 3: True
The multiverse analysis completed without any errors.


After this, the results from all universes can be loaded as a dict of dicts (default) or as a DataFrame:

In [18]:
from matplotlib import pyplot as plt

results = mverse.get_results()
print("Multiverse results structure:\n")
print(results.keys())
print(results["universe_1"].keys())
print(results["universe_1"]["graph_measure"][:5])  # Print the first 5 graph measures from universe 1

Multiverse results structure:

dict_keys(['universe_1', 'universe_2'])
dict_keys(['graph_measure', 'number', '__decisions'])
[np.float64(0.517973602484472), np.float64(0.5365295031055901), np.float64(0.531832298136646), np.float64(0.517973602484472), np.float64(0.522554347826087)]


In [6]:
results = mverse.get_results(as_df=True)
results

,universe,graph_measure,number,__decisions
1,universe_1,"[0.517973602484472, 0.5365295031055901, 0.5318...",3.14,"{'Decision 1': 'strings', 'Value 1': 'Hello', ..."
2,universe_2,"[0.7096208416867087, 0.7107480051537015, 0.708...",3.14,"{'Decision 1': 'strings', 'Value 1': 'Hello', ..."


Derived measures can be added to existing multiverse results:

In [7]:
number2x = results["number"] * 2

results = mverse.add_results("number2x", number2x)
results

,universe,graph_measure,number,__decisions,number2x
1,universe_1,"[0.517973602484472, 0.5365295031055901, 0.5318...",3.14,"{'Decision 1': 'strings', 'Value 1': 'Hello', ...",6.28
2,universe_2,"[0.7096208416867087, 0.7107480051537015, 0.708...",3.14,"{'Decision 1': 'strings', 'Value 1': 'Hello', ...",6.28


And also removed:

In [8]:
results = mverse.remove_results("number2x")
results

,universe,graph_measure,number,__decisions
1,universe_1,"[0.517973602484472, 0.5365295031055901, 0.5318...",3.14,"{'Decision 1': 'strings', 'Value 1': 'Hello', ..."
2,universe_2,"[0.7096208416867087, 0.7107480051537015, 0.708...",3.14,"{'Decision 1': 'strings', 'Value 1': 'Hello', ..."
